# DSPy Cognitive Reasoning Pipeline
Streaming-first durable evaluation notebook.
Uses `DummyLM` by default so it runs without external API keys.

In [1]:
import csv
import json
import random
from collections import Counter, defaultdict
from pathlib import Path

import dspy
from dspy.utils.dummies import DummyLM

CSV_PATH = Path('evals/cognitive_reasoning_evals.csv')
if not CSV_PATH.exists():
    CSV_PATH = Path(r'''/Users/arthurcolle/Dsco/dsco-autobot/ToolManagement/evals/cognitive_reasoning_evals.csv''')
assert CSV_PATH.exists(), f'Missing dataset: {CSV_PATH}'

evals = []
with CSV_PATH.open() as f:
    reader = csv.DictReader(f)
    for row in reader:
        evals.append(row)

print(f'Loaded {len(evals)} rows from {CSV_PATH}')
print('Difficulty split:', dict(Counter(r['difficulty'] for r in evals)))
print('Categories:', len(set(r['category'] for r in evals)))
print('Capabilities:', len(set(r['capability'] for r in evals)))


Loaded 750 rows from /Users/arthurcolle/Dsco/dsco-autobot/ToolManagement/evals/cognitive_reasoning_evals.csv
Difficulty split: {'medium': 274, 'hard': 241, 'easy': 235}
Categories: 14
Capabilities: 70


In [2]:
REASONING_METHODS = {
    'deductive': {'strength': 'certain'},
    'inductive': {'strength': 'probable'},
    'abductive': {'strength': 'plausible'},
    'analogical': {'strength': 'suggestive'},
    'causal': {'strength': 'conditional'},
    'probabilistic': {'strength': 'quantified_uncertainty'},
    'spatial': {'strength': 'geometric'},
    'temporal': {'strength': 'ordered'},
    'critical': {'strength': 'evaluative'},
    'metacognitive': {'strength': 'reflective'},
    'ethical': {'strength': 'normative'},
    'scientific': {'strength': 'empirical'},
}
CATEGORY_TO_METHOD = {
    'logical_reasoning': ['deductive', 'inductive'],
    'mathematical_reasoning': ['deductive', 'probabilistic'],
    'causal_reasoning': ['causal', 'scientific'],
    'analogical_reasoning': ['analogical'],
    'spatial_reasoning': ['spatial'],
    'temporal_reasoning': ['temporal'],
    'abstract_reasoning': ['inductive', 'deductive'],
    'linguistic_reasoning': ['critical', 'analogical'],
    'critical_thinking': ['critical', 'deductive'],
    'planning_strategy': ['causal', 'temporal'],
    'scientific_reasoning': ['scientific', 'probabilistic'],
    'social_cognition': ['abductive', 'metacognitive'],
    'metacognition': ['metacognitive', 'critical'],
    'ethical_reasoning': ['ethical', 'critical'],
}
print('Methods:', len(REASONING_METHODS), 'Category map:', len(CATEGORY_TO_METHOD))


Methods: 12 Category map: 14


In [3]:
dummy_answers = [{
    'selected_methods': 'deductive,critical',
    'method_justification': 'Selected based on category and difficulty.',
    'step_by_step_reasoning': 'Step 1: parse problem. Step 2: apply method.',
    'failure_mode_check': 'Checked for sampling bias and logical fallacies.',
    'answer': 'A',
    'confidence': 0.72,
    'approach_description': 'Structured reasoning approach.',
    'assumptions_made': 'Inputs are valid.',
    'what_could_go_wrong': 'Insufficient context.',
    'self_assessment': 'Adequate.',
    'is_correct': True,
    'error_analysis': 'None',
    'calibration_assessment': 'Slightly conservative',
    'suggested_improvement': 'Add retrieval grounding',
}] * 3000
dspy.configure(lm=DummyLM(dummy_answers))
print('Configured DSPy with DummyLM for deterministic local execution.')


Configured DSPy with DummyLM for deterministic local execution.


In [4]:
class ReasoningMethodSelector(dspy.Signature):
    prompt: str = dspy.InputField()
    category: str = dspy.InputField()
    capability: str = dspy.InputField()
    difficulty: str = dspy.InputField()
    available_methods: str = dspy.InputField()
    selected_methods: str = dspy.OutputField()
    method_justification: str = dspy.OutputField()

class ReasoningChain(dspy.Signature):
    prompt: str = dspy.InputField()
    selected_methods: str = dspy.InputField()
    answer_type: str = dspy.InputField()
    reasoning_steps: int = dspy.InputField()
    step_by_step_reasoning: str = dspy.OutputField()
    failure_mode_check: str = dspy.OutputField()
    answer: str = dspy.OutputField()
    confidence: float = dspy.OutputField()

class SelfDescribingModule(dspy.Signature):
    prompt: str = dspy.InputField()
    category: str = dspy.InputField()
    difficulty: str = dspy.InputField()
    approach_description: str = dspy.OutputField()
    assumptions_made: str = dspy.OutputField()
    what_could_go_wrong: str = dspy.OutputField()
    self_assessment: str = dspy.OutputField()

class CognitiveReasoningPipeline(dspy.Module):
    def __init__(self):
        super().__init__()
        self.select = dspy.Predict(ReasoningMethodSelector)
        self.reason = dspy.Predict(ReasoningChain)
        self.describe = dspy.Predict(SelfDescribingModule)

    def forward(self, prompt, category, capability, difficulty, answer_type, reasoning_steps):
        methods = CATEGORY_TO_METHOD.get(category, ['deductive'])
        selection = self.select(
            prompt=prompt,
            category=category,
            capability=capability,
            difficulty=difficulty,
            available_methods=json.dumps(methods),
        )
        chain = self.reason(
            prompt=prompt,
            selected_methods=selection.selected_methods,
            answer_type=answer_type,
            reasoning_steps=reasoning_steps,
        )
        description = self.describe(prompt=prompt, category=category, difficulty=difficulty)
        return dspy.Prediction(
            selected_methods=selection.selected_methods,
            method_justification=selection.method_justification,
            step_by_step_reasoning=chain.step_by_step_reasoning,
            failure_mode_check=chain.failure_mode_check,
            answer=chain.answer,
            confidence=chain.confidence,
            approach_description=description.approach_description,
            assumptions_made=description.assumptions_made,
            what_could_go_wrong=description.what_could_go_wrong,
            self_assessment=description.self_assessment,
        )

pipeline = CognitiveReasoningPipeline()
print('Pipeline ready')


Pipeline ready


In [5]:
examples = []
for row in evals:
    ex = dspy.Example(
        prompt=row['prompt'],
        category=row['category'],
        capability=row['capability'],
        difficulty=row['difficulty'],
        answer_type=row['answer_type'],
        reasoning_steps=int(row['reasoning_steps']),
        expected_answer=row['expected_answer'],
    ).with_inputs('prompt', 'category', 'capability', 'difficulty', 'answer_type', 'reasoning_steps')
    examples.append(ex)

random.seed(42)
random.shuffle(examples)
n = len(examples)
train = examples[: int(0.6 * n)]
dev = examples[int(0.6 * n) : int(0.8 * n)]
test = examples[int(0.8 * n) :]
print(f'Split: {len(train)} train / {len(dev)} dev / {len(test)} test')


Split: 450 train / 150 dev / 150 test


In [6]:
def semantic_match_metric(example, prediction, trace=None):
    expected = str(example.expected_answer).strip().lower()
    proposed = str(getattr(prediction, 'answer', '')).strip().lower()
    return 1.0 if expected[:1] == proposed[:1] else 0.0

scores = []
for ex in dev[:25]:
    pred = pipeline(
        prompt=ex.prompt,
        category=ex.category,
        capability=ex.capability,
        difficulty=ex.difficulty,
        answer_type=ex.answer_type,
        reasoning_steps=ex.reasoning_steps,
    )
    scores.append(semantic_match_metric(ex, pred))
baseline = sum(scores) / len(scores)
print(f'Baseline score (dev subset): {baseline:.3f}')


Baseline score (dev subset): 0.040


In [7]:
by_category = defaultdict(list)
for ex in test[:50]:
    pred = pipeline(
        prompt=ex.prompt,
        category=ex.category,
        capability=ex.capability,
        difficulty=ex.difficulty,
        answer_type=ex.answer_type,
        reasoning_steps=ex.reasoning_steps,
    )
    by_category[ex.category].append(semantic_match_metric(ex, pred))

print('Category analysis (test subset):')
for category in sorted(by_category.keys())[:10]:
    vals = by_category[category]
    print(f'  {category:<24} {sum(vals) / len(vals):.3f}  n={len(vals)}')


Category analysis (test subset):
  abstract_reasoning       0.143  n=7
  analogical_reasoning     0.000  n=6
  causal_reasoning         0.000  n=4
  critical_thinking        0.000  n=2
  ethical_reasoning        0.000  n=4
  linguistic_reasoning     0.000  n=4
  logical_reasoning        0.000  n=4
  mathematical_reasoning   0.000  n=4
  metacognition            0.000  n=1
  planning_strategy        0.000  n=4


In [8]:
out_dir = Path('optimized_programs')
out_dir.mkdir(parents=True, exist_ok=True)
summary = {
    'dataset_rows': len(evals),
    'train': len(train),
    'dev': len(dev),
    'test': len(test),
    'baseline_score_dev_subset': baseline,
    'reasoning_methods': list(REASONING_METHODS.keys()),
}
summary_path = out_dir / 'experiment_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print(f'Saved summary -> {summary_path}')


Saved summary -> optimized_programs/experiment_summary.json
